In [ ]:
import numpy as np
import cv2
import json
import csv
import pathlib
from pathlib import Path
from typing import List
import os
import re
import pickle
import scipy.spatial.transform
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image as IPImage


complete_corner_mapping = {
    "AB": ["A","B","D","C"],
    "BC": ["B","C","A","D"],
    "CD": ["C","D","B","A"],
    "DA": ["D","A","C","B"],
    "AD": ["A","D","B","C"],
    "DC": ["D","C","A","B"],
    "CB": ["C","B","D","A"],
    "BA": ["B","A","C","D"]
}

clockwise_rotation = ["AB","BC","CD","DA"]

default_corner_order = "BC"

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.0001)


def save_pipeline_status(outdir, step, video=None, frame=0, row=0):
    """Write a standardized status dict to `outdir/index.json`.

    Parameters:
    - outdir: path-like to the calibration output folder
    - step: string name of the pipeline step (can be arbitrary like 'detect' or 'corner_correction_in_progress')
    - video: optional video name or pair-key for stereo
    - frame: numeric frame index (int)
    - row: numeric csv row index (int)
    """
    outdir = Path(outdir)
    index_path = outdir / 'index.json'
    # Load existing index if present
    if index_path.exists():
        try:
            with index_path.open() as f:
                index = json.load(f)
        except Exception:
            index = {}
    else:
        index = {}

    index['status'] = {'step': step, 'video': video, 'frame': int(frame) if frame is not None else 0, 'row': int(row) if row is not None else 0}

    # ensure minimal structure
    index.setdefault('frame_folders', index.get('frame_folders', []))
    with index_path.open('w') as f:
        json.dump(index, f, indent=2)


def dict_from_dict_or_json_file(file_or_dict):
    if type(file_or_dict) is dict:
        return file_or_dict
    else: 
        with open(file_or_dict) as jf:
            file_content = json.load(jf)
        return file_content

# Calibration of N Cameras

**Note**: This calibration procedure requires
- A rectangular checkerboard with one recognizable side:
    ```
    A             B
      |-|-|-|-|-|
      | | | | | |
      | | | | | |
    D |-|-|-|-|-| C
      visual clue
    ```
- A manually created file `video_2_frame_number.json` which annotates for each video the frame-sequences during which the checkerboard is visible in this video:
    ```json
    {
        "cam-2": {
        "sequences": [
            [129, 344],
            [1777, 2876]
        ],
        "best_sequences_idx": [1]
        },
        "cam-3": {
            "sequences": [
                [330, 657], 
                [1355, 1552], 
                [1775, 2372], 
                [4031, 4272]
            ],
            "best_sequences_idx": [2,3] 
        },
        ...
    }
    ```
- A manually created file `filename_2_upper_side.json` which annotates during which frame-sequences which side of the checkerboard points upwards:
    ```json
    {
        "video_0.mp4": [
            {
                "start": 0,
                "end": 132,
                "upside": "BC"
            },
            ...
        ],
        ...
    }
    ```
    The specified sequences also determine which frames are gonna be used for calibration.


## Utility

### Drawing

In [ ]:
def draw_corners_with_order(image, corners):
    """Custom drawing function for detected chessboard corners."""
    corners = corners.reshape(-1, 2)  # Flatten the corners for easier indexing

    # Copy the image to draw on
    image_copy = image.copy()

    # Define font, color, and scale for drawing text
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.5
    color = (0, 0, 255)  # Red color for the text
    thickness = 1

    # Draw the corner points with their order number
    for i, corner in enumerate(corners):
        corner_position = tuple(corner.astype(int))
        cv2.circle(
            image_copy, corner_position, 5, (0, 255, 0), -1
        )  # Draw a small circle at the corner
        cv2.putText(
            image_copy,
            str(i + 1),
            corner_position,
            font,
            font_scale,
            color,
            thickness,
            cv2.LINE_AA,
        )

    return image_copy

### Detection

In [ ]:
def detect_checkerboards(outdir, video_name, cbrows, cbcols, enhance=True, slope_change_threshold_deg=15, start_row: int = 0):
    """Detect checkerboards in images from files.csv, track results in index.json.
    Uses files.csv for frame numbers and updates it with upper_side labels.
    Can resume from a given row index in files.csv by passing start_row.
    """
    outdir = Path(outdir)

    # Load index.json
    index_path = outdir / 'index.json'
    with index_path.open() as f:
        index = json.load(f)

    # Load files.csv for this video
    files_csv = Path(index['index_files'][video_name])
    files_data = []
    with files_csv.open() as f:
        reader = csv.DictReader(f)
        files_data = list(reader)

    # criteria for corner detection
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

    # prepare object points template
    objp = np.zeros((cbcols*cbrows,3), np.float32)
    objp[:,:2] = np.mgrid[0:cbrows,0:cbcols].T.reshape(-1,2)

    # Arrays to store object points and image points from all the images.
    objpoints = {} # 3d point in real world space
    imgpoints = {} # 2d points in image plane.

    prev_angle = None
    changed_frames = []  # list of (filename, frame_number, row_idx)

    def compute_angle_deg(p1, p2):
        return np.degrees(np.arctan2(p2[1] - p1[1], p2[0] - p1[0]))

    video_dir = outdir / video_name
    vis_dir = video_dir / 'visualizations'
    vis_dir.mkdir(parents=True, exist_ok=True)

    # Detection phase
    print(f"\nProcessing {len(files_data)} frames from row {start_row}...")
    for i in range(start_row, len(files_data)):
        row = files_data[i]
        imgfile = Path(row['file_loc'])
        frame_num = int(row['frame'])
        
        # Update status
        save_pipeline_status(outdir, 'detect', video=video_name, frame=frame_num, row=i)
        print(f"Processing frame {frame_num} ({i+1}/{len(files_data)})...", end='\r')

        img = cv2.imread(str(imgfile))
        if img is None:
            print(f"\nWarning: Could not read image {imgfile}")
            continue

        if enhance:
            # Apply Gaussian blur to reduce noise
            img = cv2.GaussianBlur(img, (5,5), 0)
            
            # Convert to grayscale
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            # Adaptive histogram equalization
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            gray = clahe.apply(gray)
        else:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Find the chess board corners
        ret, corners = cv2.findChessboardCorners(gray, (cbrows,cbcols), None)

        # If found, add object points, image points (after refining them)
        if ret == True:
            objpoints[imgfile.name] = [objp.tolist()]
            corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
            imgpoints[imgfile.name] = [corners2.tolist()]

            # Draw and save the corners
            img_vis = cv2.drawChessboardCorners(img.copy(), (cbrows,cbcols), corners2, ret)
            vis_path = vis_dir / f"{imgfile.name}.png"
            cv2.imwrite(str(vis_path), img_vis)

            # Get points for computing board orientation (outer edges middle points)
            points = corners2.reshape(cbrows, cbcols, 2)
            left_mid = (points[cbrows//2, 0] + points[cbrows//2-1, 0]) / 2
            right_mid = (points[cbrows//2, -1] + points[cbrows//2-1, -1]) / 2
            
            # Compute angle of the edge that's meant to face upwards
            angle = compute_angle_deg(left_mid, right_mid)
            
            # If this is a significant angle change from previous frame, record it
            if prev_angle is not None and abs(angle - prev_angle) > slope_change_threshold_deg:
                changed_frames.append((imgfile.name, frame_num, i))
            prev_angle = angle

    print(f"\nDetection complete. Found corners in {len(imgpoints)} frames.")

    # Save detected points
    objpoints_path = video_dir / 'objpoints.json'
    imgpoints_path = video_dir / 'imgpoints.json'
    
    with objpoints_path.open('w') as f:
        json.dump(objpoints, f)
    with imgpoints_path.open('w') as f:
        json.dump(imgpoints, f)

    # Update index.json to track imgpoints
    if 'imgpoints' not in index:
        index['imgpoints'] = {}
    index['imgpoints'][video_name] = str(imgpoints_path)
    this_vid_index = index['frame_folders'].index(video_name)
    if this_vid_index == len(index['frame_folders']) - 1:
        save_pipeline_status(outdir, 'corner_correction', video=index['frame_folders'][0], frame=0, row=0)
    save_pipeline_status(outdir, 'detect', video=index['frame_folders'][this_vid_index+1], frame=0, row=0)

    # Dictionary for corner order labels
    complete_corner_mapping = {"AB", "BC", "CD", "DA", "BA", "CB", "DC", "AD"}

    # Prompt user for upside labels where angle changed
    if len(changed_frames) > 0:
        print("\nDetected changes in upwards-facing edge in these frames:")
        clear_output(wait=True)  # Clear previous outputs
        
        current_upper = None
        for filename, frame_num, row_idx in changed_frames:
            vis_path = str(video_dir / 'visualizations' / f"{filename}.png")
            
            # Display image inline in notebook
            print(f"\nFrame {frame_num} (showing visualization of detected corners):")
            display(Image(filename=vis_path))
            
            user_input = input(f"Enter upwards-facing side for frame {frame_num} (e.g. 'BC' or 'AB'): ").strip().upper()
            if len(user_input) == 0:
                print("Empty input - using default 'BC'")
                user_input = "BC"
            if len(user_input) > 2:
                print(f"Warning: truncating '{user_input}' to first two letters")
                user_input = user_input[:2]
            if user_input not in complete_corner_mapping:
                print(f"Warning: '{user_input}' not in known corner mappings. Saving anyway.")

            # Fill files_data from this row until the next change with this upper side
            # We update the row at row_idx and subsequent rows until next changed_frame row
            next_idx = None
            for _, _, r in changed_frames:
                if r > row_idx:
                    next_idx = r
                    break
            end_idx = next_idx if next_idx is not None else len(files_data)
            for r in range(row_idx, end_idx):
                files_data[r]['upper_side'] = user_input

            clear_output(wait=True)  # Clear previous frame/input
                
    # Write updated files.csv with upper_side labels
    with files_csv.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(files_data[0].keys()))
        writer.writeheader()
        writer.writerows(files_data)

    return objpoints, imgpoints

In [ ]:
def enhance_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    # enhanced = cv2.equalizeHist(gray)
    # enhanced = cv2.GaussianBlur(enhanced, (5, 5), 0)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    # Sharpening
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    enhanced = cv2.filter2D(gray, -1, kernel)
    return enhanced

In [ ]:
def extract_from_video(
    videos: List[Path],
    video_2_frame_number,
    out_dir: Path,
    start_video: str = None,
    start_frame: int = 0,
    frame_step: int = 1
):
    """Extract frames for multiple videos and write index.json and files.csv.

    Can resume extraction by passing start_video and start_frame.
    The function updates index.json['status'] while running so a caller can resume later.
    Adds a 'category' column with value 'original' for extracted frames.
    """
    video_2_frame_number = dict_from_dict_or_json_file(video_2_frame_number)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    json_out_path = out_dir / "index.json"
    # If an index exists already, load it and preserve previous data
    if json_out_path.exists():
        with json_out_path.open() as f:
            json_index = json.load(f)
    else:
        json_index = {"frame_folders": [], "index_files": {}, "image_sizes": {}, "image_count": 0}

    # Ensure status field exists
    json_index.setdefault('status', {})
    video_names = [v.stem for v in videos]

    for video_path in videos:
        if not video_path.exists():
            raise Exception(f"Input video does not exist: {video_path}")
        video_name = video_path.stem
        save_pipeline_status(out_dir, 'extract', video=video_name, frame=0, row=0)

        if start_video is not None:
            if video_names.index(video_name) < video_names.index(start_video):
                print(f"skipping video {video_name} before start_video {start_video}")
                continue
        print(f"processing video: {video_path}")

        # read one frame to get vheight and vwidth
        capture = cv2.VideoCapture(str(video_path))
        try:
            if not capture.isOpened():
                raise ValueError(f"couldn't open video: {video_path}")
            success, frame = capture.read()
            if not success:
                raise ValueError(f"video {video_path} couldn't be read.")
            vheight, vwidth = frame.shape[:2]
        finally:
            capture.release()

        raw_imgs_dir = out_dir / video_name / "raw_imgs"
        raw_imgs_dir.mkdir(parents=True, exist_ok=True)

        files_csv_path = out_dir / video_name / "files.csv"
        with files_csv_path.open("w", newline="") as csv_out_file:
            csvwriter = csv.writer(
                csv_out_file, delimiter=",", quotechar='"', quoting=csv.QUOTE_MINIMAL
            )
            # Add new 'category' column. Extracted frames are 'original'.
            csvwriter.writerow(["frame", "file_loc", "stereo_partner", "upper_side", "video", "category"]) 

            capture = cv2.VideoCapture(str(video_path))

            image_count = 0
            frame_number = 0
            success = True

            try:
                if not capture.isOpened():
                    raise ValueError(f"couldn't open video: {video_path}")
                while capture.isOpened() and success:
                    success, frame = capture.read()
                    if not success:
                        break

                    # update status so we can resume if interrupted
                    save_pipeline_status(out_dir, 'extract', video=video_name, frame=frame_number, row=0)

                    if frame_number < start_frame and video_name == start_video:
                        frame_number += 1
                        continue

                    if any(
                        frame_number >= sequence[0] and frame_number <= sequence[1]
                        for sequence in video_2_frame_number[video_name]["sequences"]
                    ) and frame_number % frame_step == 0: # only process every "frame_step" frames
                        filename = f"{video_name}_{frame_number}.png"
                        abs_file_path = raw_imgs_dir / filename
                        cv2.imwrite(str(abs_file_path), frame)

                        stereo_partners = []
                        for v in video_2_frame_number:
                            if v == video_name:
                                continue
                            if any(
                                frame_number >= sequence[0] and frame_number <= sequence[1]
                                for sequence in video_2_frame_number[v]["sequences"]
                            ):
                                stereo_partners.append(v)

                        # write original frame row and set category to 'original'
                        csvwriter.writerow(
                            [frame_number, str(abs_file_path), stereo_partners, "-", video_name, "original"]
                        )
                        image_count += 1

                        print(f"frame out: {frame_number}, total image count: {image_count}",end="\r")

                    frame_number += 1

                print(f"total image: {image_count}, done")

                json_index["frame_folders"] = list(dict.fromkeys(json_index.get("frame_folders", []) + [video_name]))
                json_index["index_files"][video_name] = str(files_csv_path)
                json_index["image_sizes"][video_name] = [int(vwidth), int(vheight)]
                json_index["image_count"] = json_index.get("image_count", 0) + image_count
                # extraction for this video finished

                this_vid_index = json_index['frame_folders'].index(video_name)
                if this_vid_index != len(json_index['frame_folders']) - 1:
                    save_pipeline_status(out_dir, 'extract', video=json_index['frame_folders'][this_vid_index+1], frame=0, row=0)
                with json_out_path.open('w') as jf:
                    json.dump(json_index, jf, indent=2)
            finally:
                capture.release()

    # extraction all videos completed
    save_pipeline_status(out_dir, 'detect', video=json_index['frame_folders'][0], frame=0, row=0)
    print('Extraction complete. index.json updated at', json_out_path)

In [ ]:
"""
Identify the upwards facing side based on two criteria:
    1) It has to include the top-most corner
    2) It has to be close to horizontal
"""
def identify_upwards_facing_edge(key_corners):
    neighbours_by_index = {
        0: [key_corners[1],key_corners[2]],
        1: [key_corners[0],key_corners[3]],
        2: [key_corners[0],key_corners[3]],
        3: [key_corners[1],key_corners[2]]
    }

    # Sort by Y coordinate
    sorted_by_y = sorted(key_corners, key=lambda p: p[1])[::-1] # opencv y-coordinates are inverted
    top_most = sorted_by_y[2:4]  # Two highest corners
    top_most_corner = top_most[1]
    top_most_index = next(i for i, c in enumerate(key_corners) if np.array_equal(c, top_most_corner))

    # each edge that includes the top-most corner is a candidate for the upwards-facing side.
    facing_up_candidates = []
    if top_most[1][1] != top_most[0][1]: # if there is exactly one top most corner, i.e. the upper side is not horizontal:
        for neighbour in neighbours_by_index[top_most_index]:
            facing_up_candidates.append((top_most_corner, neighbour))
    else: # in the unlikely case that the top two corners share their y-coordinate
        facing_up_candidates.append((top_most[1], top_most[0]))
    
    def compute_angle(p1, p2):
        return np.degrees(np.arctan2(p2[1] - p1[1], p2[0] - p1[0]))

    minDeviation = 90  # Start with the worst possible case
    facing_up = None
    for candidate in facing_up_candidates:
        angle = compute_angle(*candidate)
        deviation = min(abs(angle), abs(180 - abs(angle)))  # Ensure we measure closeness to horizontal
        if deviation < minDeviation:
            facing_up = candidate
            minDeviation = deviation

    return facing_up


"""
The (arbitrarily chosen) corner-naming for the checkerboard looks like this:
A         B
  |-|-|-|
  | | | |
  | | | |
D |-|-|-| C
  loopbio

Based on coordinates of the imagepoints and based on a user-given ground-truth about positions of corners in the image, 
a correspondence between the above real-world corners and imgpoints is recovered.
"""
def detect_corner_order(imgpoints, cbwidth, cbheight, upwards_facing_side):
    # Extract four key corners
    first = imgpoints[0]
    second = imgpoints[cbheight - 1]
    third = imgpoints[cbheight * cbwidth - cbheight]
    forth = imgpoints[cbheight * cbwidth - 1]
    key_corners = np.array([first, second, third, forth])
    
    facing_up = identify_upwards_facing_edge(key_corners)
    facing_up_left  = facing_up[0] if facing_up[0][0] < facing_up[1][0] else facing_up[1]
    facing_up_right = facing_up[0] if facing_up[0][0] > facing_up[1][0] else facing_up[1]
    
    # Now, we have a correspondence between the key-corners and real-world chessboard points
    # Identify the corresponding index
    index_of_facing_up_left_end = next(
        i for i, c in enumerate(key_corners) if np.all(np.isclose(c, facing_up_left))
    )
    index_of_facing_up_right_end = next(
        i for i, c in enumerate(key_corners) if np.all(np.isclose(c, facing_up_right))
    )

    print(f"facing up left corner index: {index_of_facing_up_left_end} ({facing_up_left})\nfacing up right corner index: {index_of_facing_up_right_end} ({facing_up_right})\n")

    # Find out which of the 8 corner orders is the order of the currently detected imagepoints based on where in the order the upwards pointing corners are:
    corner_order = None
    for short, order in complete_corner_mapping.items():
        if order[index_of_facing_up_left_end] == upwards_facing_side[0] and order[index_of_facing_up_right_end] == upwards_facing_side[1]:
            corner_order = short
            
    return corner_order
    

"""
A matrix rotation of 90 degree is a transpose with reordering.
Elements that were in a row have to be into a column after rotation, hence transpose.
If we then reverse the order within the rows (i.e. we push around columns) we get each former row (now column) to it's desired rotated position.
"""
def rotate_90_clockwise(m, iterations=1):
    for i in range(iterations):
        m = np.flip(m.transpose([1,0,2]), 0)
    return m

In [ ]:
def reorder_corners(outdir, video_name, cbrows, cbcols, start_row: int = 0):
    """Reorder detected imgpoints based on upper_side labels from files.csv.
    Updates files.csv with confirmed upper_side values after reordering.
    Returns processed obj/imgpoints lists ready for calibration.
    Can resume from start_row index in files.csv.
    """
    outdir = Path(outdir)
    
    # Load index and files.csv
    with (outdir / 'index.json').open() as f:
        index = json.load(f)
    
    files_csv = Path(index['index_files'][video_name])
    files_data = []
    with files_csv.open() as f:
        reader = csv.DictReader(f)
        files_data = list(reader)

    # Load imgpoints
    imgpoints_path = Path(index['imgpoints'][video_name])
    with imgpoints_path.open() as f:
        imgpoints = json.load(f)

    # prepare object points template
    objp = np.zeros((cbrows*cbcols,3), np.float32)
    objp[:,:2] = np.mgrid[0:cbrows,0:cbcols].T.reshape(-1,2)

    # Create frame_number -> row_idx mapping for updates
    frame_to_row = {int(row['frame']): i for i, row in enumerate(files_data)}

    processed_objpoints = []
    processed_imgpoints = []

    # Process each frame in files.csv order, starting at start_row
    for idx in range(start_row, len(files_data)):
        row = files_data[idx]
        filename = os.path.splitext(os.path.basename(row['file_loc']))[0]
        if filename not in imgpoints:
            continue  # skip frames where no corners were detected
        
        points = np.array(imgpoints[filename])
        upper_side = row.get('upper_side', '-')
        
        # If no upper_side label, include points as-is for calibration
        if upper_side == '-' or not upper_side:
            imgp_ordered = points.reshape(cbrows*cbcols, 2)
        else:
            # detect current corner order and rotate/reorient as needed
            this_corner_order = detect_corner_order(points, cbheight=cbrows, cbwidth=cbcols, upwards_facing_side=upper_side)
            imgpt2 = np.copy(points).reshape(cbrows,cbcols,2)
            if this_corner_order is not None:
                if this_corner_order not in clockwise_rotation:
                    imgpt2 = np.flip(imgpt2, 1)
                    this_corner_order = this_corner_order[::-1]
                while this_corner_order != default_corner_order:
                    index_rot = clockwise_rotation.index(this_corner_order)
                    imgpt2 = rotate_90_clockwise(imgpt2, iterations=1)
                    index_rot = (index_rot-1) % len(clockwise_rotation)
                    this_corner_order = clockwise_rotation[index_rot]
                
                # Confirm the upper_side in files.csv now that we've validated it works
                frame_num = int(row['frame'])
                if frame_num in frame_to_row:
                    files_data[frame_to_row[frame_num]]['upper_side'] = upper_side
                
            imgp_ordered = imgpt2.reshape(cbrows*cbcols,2)

        processed_objpoints.append(objp.tolist())
        processed_imgpoints.append(imgp_ordered.tolist())

        # update status in index.json so pipeline can resume mid-reorder
        save_pipeline_status(outdir, 'reorder', video=video_name, frame=int(row['frame']), row=idx)

    # Save processed points
    video_dir = outdir / video_name
    proc_objpoints_path = video_dir / 'processed_objpoints.json'
    proc_imgpoints_path = video_dir / 'processed_imgpoints.json'
    
    with proc_objpoints_path.open('w') as f:
        json.dump(processed_objpoints, f)
    with proc_imgpoints_path.open('w') as f:
        json.dump(processed_imgpoints, f)

    # Update files.csv with confirmed labels
    if len(files_data) > 0:
        with files_csv.open('w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=files_data[0].keys())
            writer.writeheader()
            writer.writerows(files_data)

    # Update index.json to track processed points
    if 'imgpoints_processed' not in index:
        index['imgpoints_processed'] = {}
    index['imgpoints_processed'][video_name] = {
        'objpoints': str(proc_objpoints_path),
        'imgpoints': str(proc_imgpoints_path)
    }
    save_pipeline_status(outdir, 'calibrate', video=index['frame_folders'][0], frame=0, row=0)

    return processed_objpoints, processed_imgpoints

## Checkerboard Detection

In [ ]:
def detect_checkerboards(outdir, video_name, cbrows, cbcols, enhance=True, slope_change_threshold_deg=15, start_row: int = 0):
    """Detect checkerboards in images from files.csv, track results in index.json.
    Uses files.csv for frame numbers and updates it with upper_side labels.
    Can resume from a given row index in files.csv by passing start_row.
    Marks frames that need user labeling by setting upper_side to "!" in files.csv and
    adds 'cb_detected' category entries to files.csv for visualizations created.
    """
    outdir = Path(outdir)
    index_path = outdir / 'index.json'
    with index_path.open() as f:
        index = json.load(f)

    # Load files.csv for this video
    files_csv = Path(index['index_files'][video_name])
    files_data = []
    with files_csv.open() as f:
        reader = csv.DictReader(f)
        files_data = list(reader)

    # Ensure 'category' column exists; default missing values to 'original'
    for r in files_data:
        if 'category' not in r or not r['category']:
            r['category'] = 'original'

    # Build a queue: original rows that do not yet have a corresponding 'cb_detected' entry
    existing_cb_basenames = set()
    for r in files_data:
        if r.get('category') == 'cb_detected':
            existing_cb_basenames.add(os.path.basename(r['file_loc']))

    original_rows = [ (i, r) for i, r in enumerate(files_data) if r.get('category') == 'original' ]
    queue = []
    for i, r in original_rows:
        orig_basename = os.path.basename(r['file_loc'])
        if orig_basename not in existing_cb_basenames:
            queue.append((i, r))

    # criteria for corner detection
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

    # prepare object points template
    objp = np.zeros((cbcols*cbrows,3), np.float32)
    objp[:,:2] = np.mgrid[0:cbrows,0:cbcols].T.reshape(-1,2)

    # Arrays to store object points and image points from all the images.
    objpoints = {} # 3d point in real world space
    imgpoints = {} # 2d points in image plane.

    prev_angle = None
    changed_frames = []  # list of (filename, frame_number, row_idx)

    video_dir = outdir / video_name
    vis_dir = video_dir / 'visualizations'
    vis_dir.mkdir(parents=True, exist_ok=True)

    print(f"Found {len(queue)} original frames that need cb detection (rows in files.csv).")

    # Process the queue (only entries missing cb_detected)
    for i, row in queue:
        fpath = row['file_loc']
        frame_num = int(row['frame'])
        filename = os.path.splitext(os.path.basename(fpath))[0]

        # update status
        save_pipeline_status(outdir, 'detect', video=video_name, frame=frame_num, row=i)

        img = cv2.imread(fpath)
        if img is None:
            print(f"Could not read {fpath}; skipping")
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        if enhance:
            gray = enhance_image(gray)

        try:
            ret, corners = cv2.findChessboardCornersSB(gray, (cbrows,cbcols), None)
        except Exception:
            ret, corners = cv2.findChessboardCorners(gray, (cbrows,cbcols), None)

        if not ret:
            # No corners found: skip creating cb_detected entry but keep original row
            continue

        # corners found: refine and save
        try:
            corners2 = cv2.cornerSubPix(gray,corners, (cbrows-1,cbcols-1), (-1,-1), criteria)
        except Exception:
            corners2 = corners

        objpoints[filename] = objp.tolist()
        imgpoints[filename] = corners2.tolist()
        imgpoints[filename] = [point[0] for point in imgpoints[filename]] # remove one unnecessary dimension in the array (points were unnecessarily stored as [[x,y]] instead of [x,y])

        # Compute edge orientation and mark '!' on the original row if angle changes
        first = imgpoints[filename][0]
        second = imgpoints[filename][cbrows - 1]
        third = imgpoints[filename][cbrows * cbcols - cbrows]
        forth = imgpoints[filename][cbrows * cbcols - 1]
        key_corners = np.array([first, second, third, forth])

        facing_up = identify_upwards_facing_edge(key_corners)
        p1, p2 = map(tuple, map(np.int32, facing_up))
        try:
            angle_deg = np.degrees(np.arctan2(facing_up[1][1] - facing_up[0][1], facing_up[1][0] - facing_up[0][0]))
        except Exception:
            angle_deg = None

        if angle_deg is not None and prev_angle is not None:
            diff = abs(angle_deg - prev_angle)
            diff = min(diff, 360 - diff)
            if diff > slope_change_threshold_deg:
                # mark the ORIGINAL row for manual labeling
                files_data[i]['upper_side'] = '!'
                changed_frames.append((os.path.basename(fpath), frame_num, i))
        if angle_deg is not None:
            prev_angle = angle_deg

        # create visualization
        color_img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR) if len(gray.shape) == 2 else img.copy()
        cv2.line(color_img, p1, p2, (255, 255, 0), 2)
        first_corner = (int(corners2[0][0][0]), int(corners2[0][0][1]))
        last_corner = (int(corners2[-1][0][0]), int(corners2[-1][0][1]))
        cv2.putText(color_img, '0', first_corner, cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
        cv2.putText(color_img, str(len(corners2) - 1), last_corner, cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
        color_img = draw_corners_with_order(color_img, corners2)

        vis_name = os.path.basename(fpath)
        vis_path = vis_dir / vis_name
        cv2.imwrite(str(vis_path), color_img)

        # Append a new files.csv entry for this detection with category 'cb_detected'
        new_row = {
            'frame': row['frame'],
            'file_loc': str(vis_path),
            'stereo_partner': row.get('stereo_partner', '[]'),
            'upper_side': row.get('upper_side', '-') ,
            'video': video_name,
            'category': 'cb_detected'
        }
        files_data.append(new_row)

    # After processing queue, save detected points and write files.csv
    video_dir.mkdir(exist_ok=True)
    objpoints_path = video_dir / 'objpoints.json'
    imgpoints_path = video_dir / 'imgpoints.json'
    with objpoints_path.open('w') as f:
        json.dump(objpoints, f)
    with imgpoints_path.open('w') as f:
        json.dump(imgpoints, f)

    # Ensure header includes 'category'
    if len(files_data) > 0:
        with files_csv.open('w', newline='') as f:
            fieldnames = list(files_data[0].keys())
            # ensure consistent ordering
            if 'category' not in fieldnames:
                fieldnames.append('category')
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(files_data)

    # Update index.json to track imgpoints and mark that corner-correction may be pending
    if 'imgpoints' not in index:
        index['imgpoints'] = {}
    index['imgpoints'][video_name] = str(imgpoints_path)
    save_pipeline_status(outdir, 'corner_correction', video=index['frame_folders'][0], frame=0, row=0)

    return objpoints, imgpoints

In [ ]:
def label_upper_side(outdir, video_name, start_row: int = 0, default_label="BC"):
    outdir = Path(outdir)
    index_path = outdir / 'index.json'
    with index_path.open() as f:
        index = json.load(f)

    files_csv = Path(index['index_files'][video_name])
    with files_csv.open() as f:
        reader = csv.DictReader(f)
        files_data = list(reader)

    video_dir = outdir / video_name
    vis_dir = video_dir / 'visualizations'

    # Select rows that need user labeling: only original frames marked '!'
    marked_rows = [i for i, r in enumerate(files_data)
                   if r.get('upper_side', '') == '!' and r.get('category', '') == 'cb_detected']
    # support resume
    marked_rows = [r for r in marked_rows if r >= start_row]

    if not marked_rows:
        print(f"No frames marked for correction for {video_name} (start_row={start_row}).")
        this_vid_index = index['frame_folders'].index(video_name)
        if this_vid_index == len(index['frame_folders']) - 1:
            save_pipeline_status(outdir, 'calibrate', video=index['frame_folders'][0], frame=0, row=0)
        else:
            save_pipeline_status(outdir, 'reorder', video=index['frame_folders'][this_vid_index+1], frame=0, row=0)
        return files_csv

    print(f"\nFound {len(marked_rows)} frames to label (starting at row {start_row}).")
    complete_corner_mapping_set = {"AB","BC","CD","DA","BA","CB","DC","AD"}

    def persist_files_and_index(last_row_idx=None, in_progress=False):
        # persist CSV
        with files_csv.open('w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=files_data[0].keys())
            writer.writeheader()
            writer.writerows(files_data)
        # persist index
        if in_progress:
            save_pipeline_status(outdir, 'reorder', video=video_name, frame=0, row=last_row_idx)
        else:
            this_vid_index = index['frame_folders'].index(video_name)
            if this_vid_index == len(index['frame_folders']) - 1:
                save_pipeline_status(outdir, 'calibrate', video=index['frame_folders'][0], frame=0, row=0)
            else:
                save_pipeline_status(outdir, 'reorder', video=index['frame_folders'][this_vid_index+1], frame=0, row=0)

    for n, idx in enumerate(marked_rows, start=1):
        clear_output(wait=True)
        row = files_data[idx]
        # choose a friendly frame label: try a few possible keys
        frame_num = str(' ').join([row.get('frame_number', ''), row.get('file_loc', '')])
        print(f"[{n}/{len(marked_rows)}] Row {idx} — Frame {frame_num}")
        # pick visualization if available
        vis_path = vis_dir / os.path.basename(row.get('file_loc', ''))
        if vis_path.exists():
            display(IPImage(filename=str(vis_path)))
        else:
            imgfile = Path(row.get('file_loc', ''))
            if imgfile.exists():
                display(IPImage(filename=str(imgfile)))
            else:
                print("⚠ Image not found (showing file path):", row.get('file_loc', ''))

        # prompt user
        prompt = ("Enter upwards-facing side for this frame (e.g. BC). "
                  f"Default={default_label}. Type 'q' to quit and save progress: ")
        try:
            user_input = input(prompt).strip().upper()
        except KeyboardInterrupt:
            print("\nInterrupted by user — saving progress and exiting.")
            persist_files_and_index(last_row_idx=idx, in_progress=True)
            return files_csv

        if user_input in ('Q', 'QUIT', 'EXIT'):
            print("Quit requested. Saving progress...")
            persist_files_and_index(last_row_idx=idx, in_progress=True)
            return files_csv

        if user_input == "":
            user_input = default_label
        user_input = user_input[:2]  # keep only first two chars
        if user_input not in complete_corner_mapping_set:
            print(f"⚠ Warning: '{user_input}' not in canonical set, saving anyway.")

        # find next marked row > idx
        next_mark = next((m for m in marked_rows if m > idx), None)
        end_idx = next_mark if next_mark is not None else len(files_data)
        # assign label for rows idx .. end_idx-1
        for r in range(idx, end_idx):
            files_data[r]['upper_side'] = user_input

        # persist after each assignment
        persist_files_and_index(last_row_idx=end_idx - 1, in_progress=True)
        print(f"Saved label '{user_input}' for rows {idx}..{end_idx-1}")

    # finalize
    persist_files_and_index(in_progress=False)
    clear_output(wait=True)
    print(f"✓ Corner-order correction complete for {video_name}.")
    return files_csv

## Calibration

In [ ]:
def calibrate_camera_from_processed(outdir, video_name, image_size):
    """Calibrate a single camera from processed points and update index.json with results."""
    outdir = Path(outdir)
    
    # Load index
    with (outdir / 'index.json').open() as f:
        index = json.load(f)

    # Get paths to processed points
    proc_paths = index['imgpoints_processed'][video_name]
    with open(proc_paths['objpoints']) as f:
        objpoints_list = json.load(f)
    with open(proc_paths['imgpoints']) as f:
        imgpoints_list = json.load(f)

    # Convert to numpy arrays
    objpoints = np.array(objpoints_list, dtype=np.float32)
    imgpoints = np.array(imgpoints_list, dtype=np.float32)

    # Run calibration
    ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, tuple(image_size), None, None, criteria=criteria)

    calibration_data = {
        "video_name": video_name,
        "ret": float(ret),
        "mtx": mtx.tolist(),
        "dist": dist.tolist(),
        "rvecs": [r.tolist() for r in rvecs],
        "tvecs": [t.tolist() for t in tvecs],
    }

    # Save calibration parameters
    calib_dir = outdir / 'calibration'
    calib_dir.mkdir(exist_ok=True)
    calib_path = calib_dir / f"{video_name}_calibration.json"
    
    with calib_path.open('w') as f:
        json.dump(calibration_data, f, indent=4)

    # Update index.json to track calibration file
    if 'calib' not in index:
        index['calib'] = {}
    index['calib'][video_name] = str(calib_path)
    
    with (outdir / 'index.json').open('w') as f:
        json.dump(index, f, indent=2)

    print(f"Saved calibration for {video_name} to {calib_path}")
    return calibration_data

def stereo_calibrate(outdir, video1, video2):
    """Run stereo calibration for a pair of videos and track results in index.json."""
    outdir = Path(outdir)
    
    # Load index
    with (outdir / 'index.json').open() as f:
        index = json.load(f)

    # Load calibration parameters
    def load_calib(video):
        with open(index['calib'][video]) as f:
            return json.load(f)
    
    calib1 = load_calib(video1)
    calib2 = load_calib(video2)

    # Get processed points
    with open(index['imgpoints_processed'][video1]['objpoints']) as f:
        objpoints = np.array(json.load(f), dtype=np.float32)
    with open(index['imgpoints_processed'][video1]['imgpoints']) as f:
        imgpoints1 = np.array(json.load(f), dtype=np.float32)
    with open(index['imgpoints_processed'][video2]['imgpoints']) as f:
        imgpoints2 = np.array(json.load(f), dtype=np.float32)

    # Get image sizes
    size1 = np.array(index['image_sizes'][video1])
    size2 = np.array(index['image_sizes'][video2])

    # Extract intrinsics
    mtx1 = np.array(calib1['mtx'])
    dist1 = np.array(calib1['dist'])
    mtx2 = np.array(calib2['mtx'])
    dist2 = np.array(calib2['dist'])

    # Run stereo calibration
    retval, cameraMatrix1, distCoeffs1, cameraMatrix2, distCoeffs2, R, T, E, F = (
        cv2.stereoCalibrate(
            objpoints, imgpoints1, imgpoints2,
            mtx1, dist1, mtx2, dist2, size1[::-1],
            flags=cv2.CALIB_FIX_INTRINSIC,
        )
    )

    # Fix rotation direction
    R_flipX = cv2.Rodrigues(np.array([np.pi, 0, 0]))[0]
    R_fixed = R @ R_flipX
    R = R_fixed
    T = -T

    # Plot relationship
    r = scipy.spatial.transform.Rotation.from_matrix(R)
    angles_deg = r.as_euler("xyz", degrees=True)
    print("Rotation (degrees):", angles_deg)
    print("Translation vector:", T.ravel())

    # === Plot camera setup ===
    C1 = np.array([0, 0, 0])
    R1 = np.eye(3)
    C2 = -R.T @ T
    R2 = R

    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")
    draw_camera(ax, C1, R1, color="blue", label=f"Camera {video1}")
    draw_camera(ax, C2.ravel(), R2, color="red", label=f"Camera {video2}")
    ax.plot([C1[0], C2[0][0]], [C1[1], C2[1][0]], [C1[2], C2[2][0]], "k--", label="Baseline")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(f"Stereo Setup: {video1} - {video2}")
    ax.legend()
    ax.view_init(elev=20, azim=45)
    ax.grid(True)
    plt.tight_layout()
    plt.show()

    # Calculate projection matrices
    K1 = cameraMatrix1
    K2 = cameraMatrix2
    R1 = np.eye(3)
    t1 = np.zeros((3, 1))
    P1 = K1 @ np.hstack((R1, t1))
    P2 = K2 @ np.hstack((R2, T))

    stereo_params = {
        "video1": video1,
        "video2": video2,
        "cameraMatrix1": cameraMatrix1.tolist(),
        "distCoeffs1": distCoeffs1.tolist(),
        "cameraMatrix2": cameraMatrix2.tolist(),
        "distCoeffs2": distCoeffs2.tolist(),
        "R": R.tolist(),
        "T": T.tolist(),
        "E": E.tolist(),
        "F": F.tolist(),
        "P1": P1.tolist(),
        "P2": P2.tolist(),
    }

    # Save stereo parameters
    stereo_dir = outdir / 'stereo'
    stereo_dir.mkdir(exist_ok=True)
    stereo_path = stereo_dir / f"{video1}__{video2}_stereo.json"
    
    with stereo_path.open('w') as f:
        json.dump(stereo_params, f, indent=4)

    # Update index.json
    if 'stereo_calib' not in index:
        index['stereo_calib'] = {}
    index['stereo_calib'][f"{video1}__{video2}"] = str(stereo_path)
    
    with (outdir / 'index.json').open('w') as f:
        json.dump(index, f, indent=2)

    print(f"Stereo calibration saved to {stereo_path}")
    return stereo_params

In [ ]:
def stereo_calibrate(objpoints, imgpoints1, imgpoints2, img1_shape_np, img2_shape_np, output_dir):
    ret1, mtx1, dist1, rvecs1, tvecs1 = cv2.calibrateCamera(
        objpoints, imgpoints1, img1_shape_np[::-1], None, None
    )
    ret2, mtx2, dist2, rvecs2, tvecs2 = cv2.calibrateCamera(
        objpoints, imgpoints2, img2_shape_np[::-1], None, None
    )

    retval, cameraMatrix1, distCoeffs1, cameraMatrix2, distCoeffs2, R, T, E, F = (
        cv2.stereoCalibrate(
            objpoints,
            imgpoints1,
            imgpoints2,
            mtx1,
            dist1,
            mtx2,
            dist2,
            img1_shape_np[::-1],
            flags=cv2.CALIB_FIX_INTRINSIC,
        )
    )

    # fix rotation direction
    R_flipX = cv2.Rodrigues(np.array([np.pi, 0, 0]))[0]
    R_fixed = R @ R_flipX

    R = R_fixed
    T = -T

    # plot relationship
    r = scipy.spatial.transform.Rotation.from_matrix(R)
    angles_deg = r.as_euler("xyz", degrees=True)
    print("Rotation (degrees):", angles_deg)
    print("Translation vector:", T.ravel())

    # Camera 1 (origin)
    C1 = np.array([0, 0, 0])
    R1 = np.eye(3)
    t1 = np.zeros((3, 1))

    # Camera 2
    R2 = R
    C2 = -R2.T @ T  # Compute camera 2 center in world coords

    # === Plot both cameras ===
    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")

    draw_camera(ax, C1, R1, color="blue", label="Camera 1")
    draw_camera(ax, C2.ravel(), R2, color="red", label="Camera 2")

    # Draw baseline
    ax.plot(
        [C1[0], C2[0][0]],
        [C1[1], C2[1][0]],
        [C1[2], C2[2][0]],
        "k--",
        label="Baseline",
    )

    # Axes settings
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title("Stereo Camera Relationship")
    ax.legend()
    ax.view_init(elev=20, azim=45)
    ax.grid(True)
    plt.tight_layout()
    plt.show()

    # rectify_scale = alpha
    # R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
    #     cameraMatrix1, distCoeffs1, cameraMatrix2, distCoeffs2, enhanced_gray1.shape[::-1], R, T, alpha=rectify_scale
    # )

    # calculate stereo matrix
    # Intrinsics (from calibration)
    K1 = cameraMatrix1  # 3x3
    K2 = cameraMatrix2  # 3x3

    # Camera 1 at origin
    R1 = np.eye(3)
    t1 = np.zeros((3, 1))
    P1 = K1 @ np.hstack((R1, t1))  # 3x4

    # Camera 2 positioned relative to camera 1
    P2 = K2 @ np.hstack((R2, T))

    stereo_params = {
        "cameraMatrix1": cameraMatrix1,
        "distCoeffs1": distCoeffs1,
        "cameraMatrix2": cameraMatrix2,
        "distCoeffs2": distCoeffs2,
        "R": R,
        "T": T,
        "E": E,
        "F": F,
        "R1": R1,
        "R2": R2,
        "P1": P1,
        "P2": P2,
        # 'Q': Q,
        # 'roi1': roi1,
        # 'roi2': roi2
    }

    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "stereo_params.pickle"), "wb") as f:
        pickle.dump(stereo_params, f)

    print(
        f"Stereo calibration complete. Parameters saved to {output_dir}/stereo_params.pickle"
    )

## Usage

In [ ]:
def calibrate_videos(videos, video_2_frame_number, outdir, cbrows, cbcols, slope_change_threshold_deg=15):
    """Full calibration pipeline that can resume using index.json status.

    Behavior:
      - Steps and videos are ordered. Status indicates the (step, video[, frame/row])
        location where processing should resume (i.e. the current video still needs
        processing starting at the given frame/row).
      - The pipeline skips earlier videos/frames and resumes at that point.
      - Steps: extract -> detect -> corner_correction -> reorder -> calibrate -> stereo
    """
    outdir = Path(outdir)
    videos = [Path(v) for v in videos]
    index_path = outdir / 'index.json'

    # Define step order
    steps = ['extract', 'detect', 'corner_correction', 'reorder', 'calibrate', 'stereo']

    # Load current status
    def load_status():
        if index_path.exists():
            with index_path.open() as f:
                index = json.load(f)
            return index, index.get('status', {})
        return {}, {}

    def save_status(step, video=None, frame=0, row=0):
        """Persist a standardized status indicating the next step and video.
        When a full step completes for all videos we call this to set the next
        step and reset frame/row to 0 for the first video.
        """
        index_local, _ = load_status()
        # Ensure index exists at least as a dict
        index_local = index_local or {}
        index_local['status'] = {'step': step, 'video': video, 'frame': int(frame), 'row': int(row)}
        # write back
        with index_path.open('w') as f:
            json.dump(index_local, f, indent=2)

    def find_row_for_frame(index, video_name, frame_num):
        """Map a frame number to the row index in that video's files.csv.
        Returns the first row index whose 'frame' >= frame_num, or 0 if not found."""
        try:
            files_csv = Path(index['index_files'][video_name])
        except Exception:
            return 0
        with files_csv.open() as f:
            reader = csv.DictReader(f)
            for i, row in enumerate(reader):
                try:
                    if int(row.get('frame', -1)) >= int(frame_num):
                        return i
                except Exception:
                    continue
        return 0

    index, status = load_status()
    current_step = status.get('step')
    current_video = status.get('video')

    # Determine which step to start from
    if current_step is None:
        start_step_idx = 0
        start_video_idx = 0
    else:
        try:
            start_step_idx = steps.index(current_step)
        except ValueError:
            start_step_idx = 0
            start_video_idx = 0
        else:
            # If index already contains video ordering, start at the named video (it still needs processing)
            if 'frame_folders' in index:
                video_names = index['frame_folders']
                if current_video in video_names:
                    start_video_idx = video_names.index(current_video)
                else:
                    start_video_idx = 0
            else:
                start_video_idx = 0

    print(f"Status: step='{current_step}', video='{current_video}'")
    print(f"Resuming from step index {start_step_idx} ({steps[start_step_idx] if start_step_idx < len(steps) else 'complete'})")

    # 1) Extract frames
    if start_step_idx <= 0:
        print("\n=== Step: extract ===")
        start_video_arg = current_video if current_step == 'extract' else None
        start_frame = status.get('frame', 0) if current_step == 'extract' else 0
        extract_from_video(videos, video_2_frame_number, outdir, start_video=start_video_arg, start_frame=start_frame, frame_step=30)
        # After extract finishes for all videos, update status to next step and first video/frame
        index, _ = load_status()
        video_names = index.get('frame_folders', [])
        if video_names:
            save_status('detect', video=video_names[0], frame=0, row=0)
        start_step_idx = 1

    # Reload index to get video_names
    index, status = load_status()
    video_names = index.get('frame_folders', [])

    # If the status step is one of the subsequent steps and a video is specified,
    # start_video_idx should point to that video (it still needs processing).
    if current_step in steps[1:] and current_video in video_names:
        start_video_idx = video_names.index(current_video)

    # 2) Detect checkerboards
    if start_step_idx <= 1:
        print("\n=== Step: detect ===")
        for i, vname in enumerate(video_names):
            # Skip videos that are strictly before the start index
            if i < start_video_idx:
                print(f"Skipping {vname} (already processed)")
                continue

            # determine start_row for this video (map 'frame'->row if necessary)
            start_row = 0
            if current_step == 'detect' and vname == current_video:
                if 'row' in status:
                    start_row = int(status.get('row', 0))
                elif 'frame' in status:
                    start_row = find_row_for_frame(index, vname, status.get('frame', 0))

            if start_row > 0:
                print(f"Resuming detect for {vname} at row {start_row}")
            else:
                print(f"Running detect for {vname}")

            detect_checkerboards(str(outdir), vname, cbrows, cbcols, enhance=True,
                                 slope_change_threshold_deg=slope_change_threshold_deg, start_row=start_row)
        # After detect step completed for all videos, set status to next step and first video
        index, _ = load_status()
        video_names = index.get('frame_folders', [])
        if video_names:
            save_status('corner_correction', video=video_names[0], frame=0, row=0)
        start_step_idx = 2

    # 3) Corner-order correction (interactive)
    if start_step_idx <= 2:
        print("\n=== Step: corner_correction ===")
        index, status = load_status()
        video_names = index.get('frame_folders', [])
        # start at specified video if provided
        start_video_idx = video_names.index(current_video) if (current_step == 'corner_correction' and current_video in video_names) else 0

        for i, vname in enumerate(video_names):
            if i < start_video_idx:
                print(f"Skipping {vname} (already processed)")
                continue

            start_row = 0
            if current_step == 'corner_correction' and vname == current_video:
                if 'row' in status:
                    start_row = int(status.get('row', 0))
                elif 'frame' in status:
                    start_row = find_row_for_frame(index, vname, status.get('frame', 0))

            if start_row > 0:
                print(f"Resuming corner-order correction for {vname} at row {start_row}")
            else:
                print(f"Running corner-order correction for {vname}")

            label_upper_side(str(outdir), vname, start_row=start_row)
        # After corner correction completed for all videos, update status to next step
        index, _ = load_status()
        video_names = index.get('frame_folders', [])
        if video_names:
            save_status('reorder', video=video_names[0], frame=0, row=0)
        start_step_idx = 3

    # 4) Reorder corners
    if start_step_idx <= 3:
        print("\n=== Step: reorder ===")
        index, status = load_status()
        video_names = index.get('frame_folders', [])
        start_video_idx = video_names.index(current_video) if (current_step == 'reorder' and current_video in video_names) else 0

        for i, vname in enumerate(video_names):
            if i < start_video_idx:
                print(f"Skipping {vname} (already processed)")
                continue

            start_row = 0
            if current_step == 'reorder' and vname == current_video:
                if 'row' in status:
                    start_row = int(status.get('row', 0))
                elif 'frame' in status:
                    start_row = find_row_for_frame(index, vname, status.get('frame', 0))

            if start_row > 0:
                print(f"Resuming reorder for {vname} at row {start_row}")
            else:
                print(f"Running reorder for {vname}")

            reorder_corners(str(outdir), vname, cbrows, cbcols, start_row=start_row)
        # After reorder completed for all videos, update status to next step
        index, _ = load_status()
        video_names = index.get('frame_folders', [])
        if video_names:
            save_status('calibrate', video=video_names[0], frame=0, row=0)
        start_step_idx = 4

    # 5) Run per-video calibration
    if start_step_idx <= 4:
        print("\n=== Step: calibrate ===")
        index, status = load_status()
        video_names = index.get('frame_folders', [])
        start_video_idx = video_names.index(current_video) if (current_step == 'calibrate' and current_video in video_names) else 0

        for i, vname in enumerate(video_names):
            if i < start_video_idx:
                print(f"Skipping {vname} (already processed)")
                continue
            image_size = index.get('image_sizes', {}).get(vname)
            if image_size is None:
                print(f"No image size for {vname}; skipping")
                continue
            print(f"Running calibration for {vname}")
            calibrate_camera_from_processed(str(outdir), vname, image_size)
        # After calibrate completed for all videos, update status to stereo step
        index, _ = load_status()
        video_names = index.get('frame_folders', [])
        if video_names:
            save_status('stereo', video=video_names[0], frame=0, row=0)
        start_step_idx = 5

    # 6) Run stereo calibration for all pairs
    if start_step_idx <= 5:
        print("\n=== Step: stereo ===")
        index, status = load_status()
        video_names = index.get('frame_folders', [])
        from itertools import combinations

        all_pairs = list(combinations(video_names, 2))

        # Determine starting pair index: start at the named pair (if provided)
        start_pair_idx = 0
        if current_step == 'stereo' and current_video:
            for idx, (v1, v2) in enumerate(all_pairs):
                pair_key = f"{v1}__{v2}"
                if pair_key == current_video:
                    start_pair_idx = idx
                    break

        for idx, (v1, v2) in enumerate(all_pairs):
            if idx < start_pair_idx:
                print(f"Skipping stereo calib for {v1}-{v2} (already processed)")
                continue
            print(f"Running stereo calibration: {v1} - {v2}")
            stereo_calibrate(str(outdir), v1, v2)
        # After stereo finished for all pairs write pipeline_complete with first video/frame
        index, _ = load_status()
        video_names = index.get('frame_folders', [])
        if video_names:
            save_status('pipeline_complete', video=video_names[0], frame=0, row=0)

    # Final message
    print("\n✓ Calibration pipeline complete. Index updated.")

In [ ]:
# videos = [
#     '/media/jonathan/library/fish_data/bluegill_calib/cam-1/cam-1_15-17-14-calibration.avi',
#     '/media/jonathan/library/fish_data/bluegill_calib/cam-2/cam-2_15-17-14-calibration.avi',
#     '/media/jonathan/library/fish_data/bluegill_calib/cam-3/cam-3_15-17-14-calibration.avi',
#     '/media/jonathan/library/fish_data/bluegill_calib/cam-4/cam-4_15-17-14-calibration.avi'
# ]

videos = [
    '/mnt/c/Users/User/Documents/Studium/Bachelor/deepshapekit-v2/bluegill_data/videos/bluegill_calib/cam-1/cam-1_15-17-14-calibration.avi',
    '/mnt/c/Users/User/Documents/Studium/Bachelor/deepshapekit-v2/bluegill_data/videos/bluegill_calib/cam-2/cam-2_15-17-14-calibration.avi',
    '/mnt/c/Users/User/Documents/Studium/Bachelor/deepshapekit-v2/bluegill_data/videos/bluegill_calib/cam-3/cam-3_15-17-14-calibration.avi',
    '/mnt/c/Users/User/Documents/Studium/Bachelor/deepshapekit-v2/bluegill_data/videos/bluegill_calib/cam-4/cam-4_15-17-14-calibration.avi'
]
video_2_frame_number = '/mnt/c/Users/User/Documents/Studium/Bachelor/deepshapekit-v2/bluegill_data/videos/bluegill_calib/video_2_frame_number.json'
outdir = '/mnt/c/Users/User/Documents/Studium/Bachelor/deepshapekit-v2/calibration/calib_outputs'
cbrows, cbcols = 8, 9  # inner corner counts used for detection/calibration

calibrate_videos(videos, video_2_frame_number, outdir, cbrows, cbcols)